# Milestone 8 Graph Suitability

Aggregate-only review of Milestone 8 graph-readiness reports. This notebook reads JSON reports under `reports/` and does not open patient-level artifacts, local graph edge Parquet, raw clinical tables, or note text.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

REPORTS_ROOT = Path("..") / "reports"


def load_report(name: str) -> dict:
    path = REPORTS_ROOT / name
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


schema = load_report("milestone8_graph_schema.json")
suitability = load_report("milestone8_graph_suitability.json")
ablation = load_report("milestone8_ablation_plan.json")

assert schema["data_safety"]["report_contains_patient_rows"] is False
assert suitability["data_safety"]["report_contains_patient_rows"] is False
assert ablation["data_safety"]["report_contains_patient_rows"] is False

print("schema status:", schema["status"])
print("suitability status:", suitability["status"])
print("gate:", suitability["gate_review"]["result"])
print("next:", suitability["gate_review"]["next_action"])

In [ ]:
edge_counts = pd.DataFrame(suitability["edge_counts_by_relation"])
node_counts = pd.DataFrame(suitability["node_counts"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if not edge_counts.empty:
    edge_counts.plot.bar(x="relation_type", y="edge_count", ax=axes[0], legend=False)
    axes[0].set_title("Edges by relation")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=45)
else:
    axes[0].set_title("No graph edges")

if not node_counts.empty:
    node_counts.plot.bar(x="node_type", y="node_count", ax=axes[1], legend=False)
    axes[1].set_title("Nodes by type")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].set_title("No graph nodes")

plt.tight_layout()

In [ ]:
coverage = pd.DataFrame(suitability["split_coverage_and_cold_start"])
if not coverage.empty:
    display_columns = [
        "source",
        "split",
        "positive_row_count",
        "in_graph_positive_row_count",
        "positive_graph_coverage_rate",
        "candidate_medication_cold_start_rate",
        "condition_cold_start_rate",
    ]
    display(coverage[display_columns])

    ax = coverage.set_index(["source", "split"])[
        ["candidate_medication_cold_start_rate", "condition_cold_start_rate"]
    ].plot.bar(figsize=(10, 4))
    ax.set_title("Cold-start rates by split")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    plt.tight_layout()
else:
    print("No split coverage rows available.")

In [ ]:
components = suitability["connected_components"]
sparsity = pd.DataFrame(suitability["sparsity"])
print("Connected components:", components)
if not sparsity.empty:
    display(sparsity)
    ax = sparsity.plot.bar(
        x="relation_type", y="density", figsize=(10, 4), legend=False
    )
    ax.set_title("Relation density")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()